In [63]:
import json
import numpy as np

In [64]:
INPUT_PATH = "../data/processed_papers/embedded_chunks.json"

with open(
    INPUT_PATH,
    "r",
    encoding="utf-8"
) as f:
    chunks = json.load(f)

print(len(chunks))

643


In [65]:
# Extract Embeddings

embeddings = np.array(
    [chunk["embedding"] for chunk in chunks],
    dtype=np.float32
)

print(embeddings.shape)

(643, 384)


In [66]:
#Install FAISS
!pip install faiss-gpu-cu12

Access is denied.


In [67]:
import faiss

print("FAISS Loaded Successfully")
print(faiss.__version__)

FAISS Loaded Successfully
1.14.2


In [68]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

faiss.normalize_L2(embeddings)

index.add(embeddings)

print("Vectors in index:", index.ntotal)

Vectors in index: 643


In [69]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda"
)

print("Query Embedding Model Loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Query Embedding Model Loaded


In [70]:
print(model.device)

cuda:0


Dense Retrieval Function

In [71]:
def dense_search(
    query,
    top_k=5
):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype(np.float32)

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search FAISS index
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({
            "score": float(score),
            "chunk": chunks[idx]
        })

    return results

In [72]:
results = dense_search(
    "What is stochastic volatility?"
)

print(len(results))

5


In [73]:
for i, result in enumerate(results):

    print("\n" + "="*80)
    print(f"RESULT {i+1}")
    print("="*80)

    print("Score:", result["score"])

    print("\nPaper:")
    print(result["chunk"]["paper_title"])

    print("\nText:")
    print(result["chunk"]["text"][:500])


RESULT 1
Score: 0.7970714569091797

Paper:
Option pricing under non-Markovian stochastic volatility models: A deep

Text:
 stochastic diﬀerential equati on, allowing the application of standard analytical tools. We p ropose a deep signature approach for both linear and nonlinear representations and rigorously prove the conver gence of the algorithm. Numerical examples demonstrate the eﬀectiveness of our approach for both Markovian and non- Markovian volatility models, oﬀering a theoretically grounded and computationally eﬃcient framework for option pricing. 1 Introduction The pricing problem lies at the heart of ma

RESULT 2
Score: 0.7842767238616943

Paper:
Stochastic Volatility, Jumps, and Rates: A Unified Framework for

Text:
Stochastic Volatility, Jumps, and Rates: A Unified Framework for Option Pricing and Term-Structure Simulation Nunik Srikandi Putri1, Ajay Kumar Verma2, Neo Paul Lesupi3 1Independent Researcher (Aenimatica Tech), 2Independent Researcher, 3Independent Researcher

In [74]:
import faiss
import os

os.makedirs("../data/vector_store", exist_ok=True)

faiss.write_index(
    index,
    "../data/vector_store/faiss_index.bin"
)

print("FAISS Index Saved")

FAISS Index Saved


In [75]:
loaded_index = faiss.read_index(
    "../data/vector_store/faiss_index.bin"
)

print("Vectors:", loaded_index.ntotal)

Vectors: 643


In [76]:
#Import BM25
from rank_bm25 import BM25Okapi

print("BM25 Loaded Successfully")

BM25 Loaded Successfully


In [77]:
import re

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_chunks = [
    tokenize(chunk["text"])
    for chunk in chunks
]

In [78]:
bm25 = BM25Okapi(tokenized_chunks)

print("BM25 Index Created")

BM25 Index Created


In [79]:
#BM25 Search Function
def bm25_search(query, top_k=5):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = scores.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:

        results.append({
            "score": float(scores[idx]),
            "chunk": chunks[idx]
        })

    return results

In [80]:
results = bm25_search(
    "stochastic volatility",
    top_k=5
)

print(len(results))

5


In [81]:
results = bm25_search(
    "stochastic volatility",
    top_k=5
)

print(len(results))

5


In [82]:
for i, result in enumerate(results):

    print("\n" + "="*80)
    print(f"BM25 RESULT {i+1}")
    print("="*80)

    print("Score:", result["score"])

    print("\nPaper:")
    print(result["chunk"]["paper_title"])

    print("\nText:")
    print(result["chunk"]["text"][:500])


BM25 RESULT 1
Score: 7.279156178887711

Paper:
Stochastic Volatility, Jumps, and Rates: A Unified Framework for

Text:
consistent with recent findings on calibration stability in stochastic volatility models (Agazzotti et al., 2025). For 60 -day maturities, the Bates extension yields jump intensities close to zero, aligning with recent evidence that short -term implied- volatility structures can often be captured without substantial jump components (Agazzotti et al., 2025). The CIR model calibrated to the Euribor term structure produces economically intuitive parameters and realistic forward -rate scenarios, con

BM25 RESULT 2
Score: 7.197340893733335

Paper:
Stochastic Volatility, Jumps, and Rates: A Unified Framework for

Text:
ng the model to Bates shows that jump intensities converge to values eff ectively equal to zero for 60-day maturities, echoing empirical findings that jumps contribute marginally to short -term smile fitting. We further compare our calibration approach with t

In [88]:
#RRF Function
from collections import defaultdict

def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    k=60
):

    fused_scores = defaultdict(float)

    # Dense results
    for rank, result in enumerate(dense_results):

        chunk_id = result["chunk"]["global_chunk_id"]

        fused_scores[chunk_id] += 1 / (k + rank + 1)

    # BM25 results
    for rank, result in enumerate(bm25_results):

        chunk_id = result["chunk"]["global_chunk_id"]

        fused_scores[chunk_id] += 1 / (k + rank + 1)

    return fused_scores

In [105]:
#Hybrid search function
# Hybrid Search Function

def hybrid_search_rrf(
    query,
    top_k=5
):

    dense_results = dense_search(
        query,
        top_k=50
    )

    bm25_results = bm25_search(
        query,
        top_k=50
    )

    fused_scores = reciprocal_rank_fusion(
        dense_results,
        bm25_results
    )

    ranked_ids = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    final_results = []

    for chunk_id, score in ranked_ids[:top_k]:

        final_results.append({
            "score": score,
            "chunk": chunks[chunk_id]
        })

    return final_results

In [106]:
results = hybrid_search_rrf(
    "stochastic volatility"
)

print(len(results))

5


In [107]:
for i, result in enumerate(results):

    print("\n" + "="*80)
    print(f"HYBRID RESULT {i+1}")
    print("="*80)

    print("Score:", result["score"])

    print("\nPaper:")
    print(result["chunk"]["paper_title"])

    print("\nText:")
    print(result["chunk"]["text"][:500])


HYBRID RESULT 1
Score: 0.031544957774465976

Paper:
Stochastic Volatility, Jumps, and Rates: A Unified Framework for

Text:
consistent with recent findings on calibration stability in stochastic volatility models (Agazzotti et al., 2025). For 60 -day maturities, the Bates extension yields jump intensities close to zero, aligning with recent evidence that short -term implied- volatility structures can often be captured without substantial jump components (Agazzotti et al., 2025). The CIR model calibrated to the Euribor term structure produces economically intuitive parameters and realistic forward -rate scenarios, con

HYBRID RESULT 2
Score: 0.031009615384615385

Paper:
Option pricing under non-Markovian stochastic volatility models: A deep

Text:
Option pricing under non-Markovian stochastic volatility models: A deep signature approach Jingtang Ma∗ Xianglin Wu† Wenyuan Li‡ May 29, 2026 Abstract This paper studies the option pricing problem in which the un derlying asset follows a non-

In [108]:
print(chunks[0].keys())

dict_keys(['global_chunk_id', 'local_chunk_id', 'paper_title', 'filename', 'text', 'embedding'])


In [109]:
print(chunks[0]["global_chunk_id"])

for i in range(20):
    print(
        chunks[i]["paper_title"],
        chunks[i]["global_chunk_id"]
    )

0
Option pricing under non-Markovian stochastic volatility models: A deep 0
Option pricing under non-Markovian stochastic volatility models: A deep 1
Option pricing under non-Markovian stochastic volatility models: A deep 2
Option pricing under non-Markovian stochastic volatility models: A deep 3
Option pricing under non-Markovian stochastic volatility models: A deep 4
Option pricing under non-Markovian stochastic volatility models: A deep 5
Option pricing under non-Markovian stochastic volatility models: A deep 6
Option pricing under non-Markovian stochastic volatility models: A deep 7
Option pricing under non-Markovian stochastic volatility models: A deep 8
Option pricing under non-Markovian stochastic volatility models: A deep 9
Option pricing under non-Markovian stochastic volatility models: A deep 10
Option pricing under non-Markovian stochastic volatility models: A deep 11
Option pricing under non-Markovian stochastic volatility models: A deep 12
Option pricing under non-Markovia

In [110]:
titles = {}

for chunk in chunks:
    cid = chunk["global_chunk_id"]

    if cid in titles:
        print("DUPLICATE FOUND:")
        print(cid)
        print(titles[cid])
        print(chunk["paper_title"])
        break

    titles[cid] = chunk["paper_title"]

In [111]:
print(chunks[0].keys())

dict_keys(['global_chunk_id', 'local_chunk_id', 'paper_title', 'filename', 'text', 'embedding'])


In [112]:
results = hybrid_search_rrf(
    "stochastic volatility"
)

print(len(results))

5


In [113]:
for i, result in enumerate(results):

    print(i+1, result["chunk"]["paper_title"])

1 Stochastic Volatility, Jumps, and Rates: A Unified Framework for
2 Option pricing under non-Markovian stochastic volatility models: A deep
3 Stochastic Volatility, Jumps, and Rates: A Unified Framework for
4 Option pricing under non-Markovian stochastic volatility models: A deep
5 Stochastic Volatility, Jumps, and Rates: A Unified Framework for


In [114]:
for i, result in enumerate(results):

    print(
        i+1,
        result["chunk"]["paper_title"],
        result["chunk"]["global_chunk_id"],
        result["chunk"]["local_chunk_id"]
    )

1 Stochastic Volatility, Jumps, and Rates: A Unified Framework for 357 35
2 Option pricing under non-Markovian stochastic volatility models: A deep 0 0
3 Stochastic Volatility, Jumps, and Rates: A Unified Framework for 323 1
4 Option pricing under non-Markovian stochastic volatility models: A deep 1 1
5 Stochastic Volatility, Jumps, and Rates: A Unified Framework for 326 4


In [115]:
results = hybrid_search_rrf(
    "garch forecasting"
)

for i, result in enumerate(results):
    print(
        i+1,
        result["chunk"]["paper_title"]
    )

1 Deep Learning Forecasting of the U.S. Aggregate Bond Index
2 Multivariate Financial Forecasting using the
3 Multivariate Financial Forecasting using the
4 Multivariate Financial Forecasting using the
5 Multivariate Financial Forecasting using the


In [116]:
results = hybrid_search_rrf(
    "market efficiency"
)

for i, result in enumerate(results):
    print(
        i+1,
        result["chunk"]["paper_title"]
    )

1 QUANTS AND MARKET EFFICIENCY
2 QUANTS AND MARKET EFFICIENCY
3 QUANTS AND MARKET EFFICIENCY
4 QUANTS AND MARKET EFFICIENCY
5 QUANTS AND MARKET EFFICIENCY


In [117]:
results = hybrid_search_rrf(
    "stochastic volatility option pricing"
)

for i, result in enumerate(results):
    print(
        i+1,
        result["chunk"]["paper_title"]
    )

1 Option pricing under non-Markovian stochastic volatility models: A deep
2 Stochastic Volatility, Jumps, and Rates: A Unified Framework for
3 Option pricing under non-Markovian stochastic volatility models: A deep
4 Option pricing under non-Markovian stochastic volatility models: A deep
5 Stochastic Volatility, Jumps, and Rates: A Unified Framework for


In [118]:
import pickle

with open("../data/hybrid_candidates.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved")

Saved
